# Análise de uma corrida (OpenF1 API)

Este notebook consome o pipeline `f1.pipelines.race.build_race_dataset` **em memória**,
sem nenhuma etapa de persistência (ver Fase 5 do `plan.md`), e produz uma primeira
análise exploratória de uma sessão de corrida: ritmo de volta por piloto, evolução
de posições, condições climáticas e resultado final.

Troque o `SESSION_KEY` abaixo pela sessão que deseja analisar. O valor padrão
(`9222`) é a corrida do GP da Bélgica de 2023, usada como exemplo em `docs/api.md`.

In [ ]:
import matplotlib.pyplot as plt

from f1.pipelines.race import build_race_dataset

SESSION_KEY = 9222

dataset = build_race_dataset(SESSION_KEY)
driver_label = {
    driver.driver_number: driver.name_acronym or str(driver.driver_number)
    for driver in dataset.drivers
}
dataset.session

## Ritmo de volta por piloto

Duração de cada volta (em segundos) ao longo da corrida, por piloto. Voltas de
saída do pit lane (`is_pit_out_lap=True`) e voltas sem `lap_duration` (ex.: última
volta incompleta) são descartadas para não distorcer a escala do gráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for driver_number in driver_label:
    laps = sorted(
        (
            lap
            for lap in dataset.laps
            if lap.driver_number == driver_number
            and lap.lap_duration is not None
            and not lap.is_pit_out_lap
        ),
        key=lambda lap: lap.lap_number,
    )
    if not laps:
        continue
    ax.plot(
        [lap.lap_number for lap in laps],
        [lap.lap_duration for lap in laps],
        marker="o",
        markersize=2,
        label=driver_label[driver_number],
    )

ax.set_xlabel("Volta")
ax.set_ylabel("Duração da volta (s)")
ax.set_title("Ritmo de volta por piloto")
ax.legend(loc="upper right", fontsize="small", ncol=2)
plt.show()

## Evolução de posições ao longo da corrida

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for driver_number in driver_label:
    positions = sorted(
        (pos for pos in dataset.positions if pos.driver_number == driver_number),
        key=lambda pos: pos.date,
    )
    if not positions:
        continue
    ax.plot(
        [pos.date for pos in positions],
        [pos.position for pos in positions],
        label=driver_label[driver_number],
    )

ax.set_xlabel("Horário")
ax.set_ylabel("Posição")
ax.set_title("Evolução de posições")
ax.invert_yaxis()
ax.legend(loc="upper right", fontsize="small", ncol=2)
fig.autofmt_xdate()
plt.show()

## Condições climáticas

In [ ]:
weather = sorted(dataset.weather, key=lambda w: w.date)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    [w.date for w in weather],
    [w.air_temperature for w in weather],
    label="Temperatura do ar (°C)",
)
ax.plot(
    [w.date for w in weather],
    [w.track_temperature for w in weather],
    label="Temperatura da pista (°C)",
)
ax.set_xlabel("Horário")
ax.set_ylabel("Temperatura (°C)")
ax.set_title("Condições climáticas durante a sessão")
ax.legend()
fig.autofmt_xdate()
plt.show()

## Resultado final

In [ ]:
results = sorted(dataset.results, key=lambda r: (r.position is None, r.position or 0))

for result in results:
    status = (
        "DNF" if result.dnf else "DSQ" if result.dsq else "DNS" if result.dns else ""
    )
    print(
        f"{result.position or '-':>3}  "
        f"{driver_label.get(result.driver_number, result.driver_number):>4}  "
        f"voltas={result.number_of_laps}  {status}"
    )